In [ ]:
import os
import cv2
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, Conv2DTranspose, Concatenate, BatchNormalization, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import LearningRateScheduler, ModelCheckpoint, EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [ ]:
def dice_coef(y_true, y_pred, smooth=1):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth)

def iou_coef(y_true, y_pred, smooth=1):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    union = tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)


In [ ]:
# Function to split images into tiles
def split_image_into_tiles(image_path, mask_path, tile_size, size):
    img = tifffile.imread(image_path)
    mask = tifffile.imread(mask_path)
    mask = mask[:, :, 0] if len(mask.shape) == 3 else mask

    tiles_img, tiles_mask = [], []
    for x in range(0, img.shape[1], tile_size):
        for y in range(0, img.shape[0], tile_size):
            tile_img = img[y:y+tile_size, x:x+tile_size, :]
            tile_mask = mask[y:y+tile_size, x:x+tile_size]

            tile_img = cv2.resize(tile_img, (size, size))
            tile_mask = cv2.resize(tile_mask, (size, size))
            tile_mask = (tile_mask > 0).astype(np.uint8)

            tiles_img.append(tile_img)
            tiles_mask.append(tile_mask)

    return np.array(tiles_img), np.array(tiles_mask)

# Load dataset
def load_data(image_dir, mask_dir, tile_size=256, size=256):
    images, masks = [], []
    image_filenames = sorted(os.listdir(image_dir))
    mask_filenames = sorted(os.listdir(mask_dir))

    for image_filename in image_filenames:
        if image_filename.endswith(".TIF"):
            mask_filename = image_filename.replace(".TIF", "_mask.TIF")
            if mask_filename in mask_filenames:
                img_path = os.path.join(image_dir, image_filename)
                mask_path = os.path.join(mask_dir, mask_filename)
                img, mask = split_image_into_tiles(img_path, mask_path, tile_size, size)
                images.extend(img)
                masks.extend(mask)
    return np.array(images), np.array(masks)

# Paths
image_dir = "../../datasets/images"
mask_dir = "../../datasets/masks"
size = 256

# Load data
tiles_img, tiles_mask = load_data(image_dir, mask_dir, tile_size=size, size=size)

# Split sets
X_train, X_test, y_train, y_test = train_test_split(tiles_img, tiles_mask, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

# Reshape masks
y_train = y_train[..., np.newaxis]
y_val = y_val[..., np.newaxis]
y_test = y_test[..., np.newaxis]


In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, Conv2DTranspose, Concatenate
from tensorflow.keras.optimizers import Adam


# Define complex VGG16 U-Net model
def vgg16_unet_model(input_size=(size, size, 3), freeze_encoder=True):
    # Use VGG16 as the backbone
    vgg16_base = VGG16(weights='imagenet', include_top=False, input_shape=input_size)

    # Encoder
    encoder_output = vgg16_base.get_layer('block5_pool').output
    
    # Freeze the layers in the encoder
    if freeze_encoder:
        for layer in vgg16_base.layers:
            layer.trainable = False

    # Decoder
    x = Conv2DTranspose(512, (2, 2), strides=(2, 2), padding='same')(encoder_output)
    x = Concatenate()([x, vgg16_base.get_layer('block4_pool').output])
    x = Conv2D(512, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x = Conv2DTranspose(512, (2, 2), strides=(2, 2), padding='same')(x)
    x = Concatenate()([x, vgg16_base.get_layer('block3_pool').output])
    x = Conv2D(512, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x = Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(x)
    x = Concatenate()([x, vgg16_base.get_layer('block2_pool').output])
    x = Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x = Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(x)
    x = Concatenate()([x, vgg16_base.get_layer('block1_pool').output])
    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    # Output layer
    output = Conv2D(1, (1, 1), activation='sigmoid', padding='same')(x)

    # Resize the output to match the size of the ground truth masks
    output = tf.image.resize(output, (size, size), method='bilinear')

    model = Model(inputs=vgg16_base.input, outputs=output)

    #model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model.compile(
    optimizer=Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', dice_coef, iou_coef])

    return model

# Usage
size = 256
model = vgg16_unet_model(input_size=(size, size, 3), freeze_encoder=True)
model.summary()


In [ ]:
# LR schedule
def lr_schedule(epoch):
    initial_lr = 1e-4
    decay = 0.9
    return initial_lr * (decay ** (epoch // 10))

lr_scheduler = LearningRateScheduler(lr_schedule)

# Data augmentation
datagen = ImageDataGenerator(rescale=1./255,
                             shear_range=0.2,
                             zoom_range=0.2,
                             horizontal_flip=True,
                             rotation_range=20,
                             width_shift_range=0.2,
                             height_shift_range=0.2,
                             brightness_range=[0.8, 1.2])

# Callbacks
checkpointer = ModelCheckpoint("best_vgg16_unet.h5", monitor="val_dice_coef", mode="max",
                               save_best_only=True, verbose=1)
earlyStopping = EarlyStopping(monitor="val_dice_coef", patience=5, mode="max", verbose=1)


In [ ]:
history = model.fit(datagen.flow(X_train, y_train, batch_size=32),
                    validation_data=(X_val/255.0, y_val),
                    epochs=50,
                    callbacks=[lr_scheduler, earlyStopping, checkpointer])


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras import backend as K
from tensorflow.keras.metrics import Precision, Recall

# ----------------------------
# Define your custom metrics again
# ----------------------------
def dice_coef(y_true, y_pred, smooth=1):
    y_pred = K.cast(y_pred > 0.5, "float32")
    intersection = K.sum(y_true * y_pred)
    return (2. * intersection + smooth) / (K.sum(y_true) + K.sum(y_pred) + smooth)

def iou_coef(y_true, y_pred, smooth=1):
    y_pred = K.cast(y_pred > 0.5, "float32")
    intersection = K.sum(y_true * y_pred)
    union = K.sum(y_true) + K.sum(y_pred) - intersection
    return (intersection + smooth) / (union + smooth)

def mean_iou(y_true, y_pred):
    y_pred = K.cast(y_pred > 0.5, "float32")
    intersect = K.sum(y_true * y_pred)
    union = K.sum(y_true) + K.sum(y_pred) - intersect
    return intersect / (union + K.epsilon())

# ----------------------------
# Load the saved model
# ----------------------------
model = load_model(
    "best_vgg16_unet.h5",
    custom_objects={
        "dice_coef": dice_coef,
        "iou_coef": iou_coef,
        "mean_iou": mean_iou
    }
)

# ----------------------------
# Recompile with extra metrics
# ----------------------------
model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy", dice_coef, iou_coef, Precision(), Recall(), mean_iou])

# ----------------------------
# Evaluate on your validation/test set
# ----------------------------
# Replace X_val, y_val with your actual validation or test data
results = model.evaluate(X_val, y_val, verbose=1)

print("\n📊 Evaluation Results:")
for name, value in zip(model.metrics_names, results):
    print(f"{name}: {value*100:.2f}%")


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils import resample



def bootstrap_confidence_interval(y_true, y_pred, metric_fn, n_bootstraps=100, alpha=0.95):
    """Bootstrap CI + std for a given metric"""
    stats = []
    n = len(y_true)
    for _ in range(n_bootstraps):
        indices = np.random.randint(0, n, n)
        if metric_fn.__name__ == "roc_auc_score":  # ROC-AUC requires probs
            stat = metric_fn(y_true[indices], y_pred[indices])
        else:  # Binary metrics
            stat = metric_fn(y_true[indices], y_pred[indices])
        stats.append(stat)
    
    stats = np.array(stats)
    mean_val = np.mean(stats)
    std_val  = np.std(stats)
    lower = np.percentile(stats, ((1 - alpha) / 2) * 100)
    upper = np.percentile(stats, (alpha + (1 - alpha) / 2) * 100)
    
    return mean_val, std_val, (lower, upper)


# --- Evaluate model ---
loss, acc, dice, iou = model.evaluate(X_test/255.0, y_test)
print(f"Test Loss: {loss:.4f}, Accuracy: {acc:.4f}, Dice: {dice:.4f}, IoU: {iou:.4f}")

# --- Predictions ---
y_pred = model.predict(X_test/255.0)
y_pred_bin = (y_pred > 0.5).astype(np.uint8)

# Flatten
y_true_flat = y_test.flatten()
y_pred_flat = y_pred_bin.flatten()
y_pred_probs = y_pred.flatten()

# --- Metrics ---
metrics = {
    "Accuracy": lambda yt, yp: np.mean(yt == yp),
    "Precision": lambda yt, yp: precision_score(yt, yp),
    "Recall": lambda yt, yp: recall_score(yt, yp),
    "F1-score": lambda yt, yp: f1_score(yt, yp),
    "ROC-AUC": lambda yt, yp: roc_auc_score(yt, yp),
    "Dice": lambda yt, yp: (2*np.sum(yt*yp))/(np.sum(yt)+np.sum(yp)+1e-7),
    "IoU": lambda yt, yp: np.sum(yt*yp)/(np.sum(yt)+np.sum(yp)-np.sum(yt*yp)+1e-7)
}

print("\n📊 Metrics with 95% Confidence Intervals:")
for name, fn in metrics.items():
    if name == "ROC-AUC":
        mean_val, std_val, (low, high) = bootstrap_confidence_interval(y_true_flat, y_pred_probs, fn)
    else:
        mean_val, std_val, (low, high) = bootstrap_confidence_interval(y_true_flat, y_pred_flat, fn)
    print(f"{name}: {mean_val:.4f} ± {std_val:.4f}  (95% CI: {low:.4f} – {high:.4f})")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Flatten ground truth and predictions
y_true_flat = y_test.flatten()
y_pred_flat = (y_pred.flatten() > 0.5).astype(int)  # threshold at 0.5

# Confusion Matrix
cm = confusion_matrix(y_true_flat, y_pred_flat)
print("Confusion Matrix:\n", cm)

# Classification Report
print(classification_report(y_true_flat, y_pred_flat))

# Heatmap
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Non-Forest","Forest"],
            yticklabels=["Non-Forest","Forest"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
# Plot training history for loss, accuracy, dice, iou
plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')

plt.xlabel('Epoch')
plt.ylabel('Metrics')
plt.title('vgg16_U-Net Training History')
plt.legend()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Epochs: 1 to 20 (early stopping at 20)
epochs = range(1, 21)

# Training metrics
train_loss = [
    0.5790, 0.4441, 0.4018, 0.3549, 0.3402, 0.3271, 0.3100, 0.3070, 0.2991, 0.3012,
    0.2989, 0.2954, 0.3004, 0.2934, 0.2939, 0.2912, 0.2904, 0.2873, 0.2908, 0.2925
]

train_accuracy = [
    0.7654, 0.8262, 0.8443, 0.8538, 0.8573, 0.8612, 0.8681, 0.8660, 0.8688, 0.8662,
    0.8679, 0.8695, 0.8661, 0.8707, 0.8691, 0.8699, 0.8708, 0.8717, 0.8705, 0.8694
]

train_dice = [
    0.5826, 0.6883, 0.7289, 0.7688, 0.7852, 0.7952, 0.8087, 0.8156, 0.8209, 0.8249,
    0.8188, 0.8258, 0.8241, 0.8261, 0.8299, 0.8293, 0.8293, 0.8310, 0.8282, 0.8286
]

train_iou = [
    0.4122, 0.5256, 0.5742, 0.6253, 0.6478, 0.6615, 0.6810, 0.6903, 0.6985, 0.7039,
    0.6961, 0.7047, 0.7028, 0.7056, 0.7108, 0.7103, 0.7102, 0.7129, 0.7086, 0.7093
]

# Validation metrics
val_loss = [
    0.7033, 0.4845, 0.6945, 0.5323, 0.2758, 0.3805, 0.2395, 0.2706, 0.2318, 0.2699,
    0.2370, 0.2417, 0.2259, 0.2322, 0.2262, 0.2269, 0.2326, 0.3507, 0.2250, 0.2340
]

val_accuracy = [
    0.7929, 0.8106, 0.7925, 0.7942, 0.8963, 0.8242, 0.9043, 0.8757, 0.9035, 0.8841,
    0.9074, 0.8935, 0.9084, 0.9019, 0.9090, 0.8989, 0.9066, 0.8017, 0.9111, 0.9069
]

val_dice = [
    0.5910, 0.6995, 0.6955, 0.7266, 0.8081, 0.7767, 0.8412, 0.8303, 0.8495, 0.8381,
    0.8457, 0.8480, 0.8545, 0.8526, 0.8616, 0.8556, 0.8495, 0.8218, 0.8500, 0.8425
]

val_iou = [
    0.4205, 0.5391, 0.5351, 0.5724, 0.6797, 0.6362, 0.7276, 0.7111, 0.7399, 0.7237,
    0.7347, 0.7373, 0.7479, 0.7446, 0.7584, 0.7491, 0.7403, 0.7013, 0.7411, 0.7294
]

In [ ]:
plt.figure(figsize=(16, 12))

# Plot 1: Loss
plt.subplot(2, 2, 1)
plt.plot(epochs, train_loss, 'bo-', label='Training Loss', linewidth=2)
plt.plot(epochs, val_loss, 'r-o', label='Validation Loss', linewidth=2)
plt.title('Training and Validation Loss', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 2: Accuracy
plt.subplot(2, 2, 2)
plt.plot(epochs, train_accuracy, 'bo-', label='Training Accuracy', linewidth=2)
plt.plot(epochs, val_accuracy, 'r-o', label='Validation Accuracy', linewidth=2)
plt.title('Training and Validation Accuracy', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 3: Dice Coefficient
plt.subplot(2, 2, 3)
plt.plot(epochs, train_dice, 'bo-', label='Training Dice Coefficient', linewidth=2)
plt.plot(epochs, val_dice, 'r-o', label='Validation Dice Coefficient', linewidth=2)
plt.title('Training and Validation Dice Coefficient', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('Dice Coefficient')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 4: IoU
plt.subplot(2, 2, 4)
plt.plot(epochs, train_iou, 'bo-', label='Training IoU', linewidth=2)
plt.plot(epochs, val_iou, 'r-o', label='Validation IoU', linewidth=2)
plt.title('Training and Validation IoU', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('IoU')
plt.legend()
plt.grid(True, alpha=0.3)

# Adjust layout and show
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Define the data
data = [
    # Model, Accuracy, Precision, Recall, F1, ROC-AUC, Dice, IoU, FN
    ["AlexNet", "0.7635\n(0.7633–0.7636)", "0.9936\n(0.9935–0.9936)", "0.5602\n(0.5601–0.5604)",
     "0.7164\n(0.7163–0.7166)", "0.7535\n(0.7533–0.7536)", "0.7165\n(0.7163–0.7166)",
     "0.5582\n(0.5580–0.5584)", "8,102,319"],

    ["DenseNet50", "0.8826\n(0.8825–0.8827)", "0.9183\n(0.9182–0.9184)", "0.8561\n(0.8560–0.8563)",
     "0.8861\n(0.8860–0.8862)", "0.9533\n(0.9533–0.9534)", "0.8861\n(0.8860–0.8862)",
     "0.7955\n(0.7954–0.7957)", "9,015,355"],

    ["UNet (basic)", "0.7641\n(0.7639–0.7642)", "1.0000\n(1.0000–1.0000)", "0.5577\n(0.5575–0.5579)",
     "0.7160\n(0.7159–0.7162)", "0.9256\n(0.9256–0.9257)", "0.7160\n(0.7159–0.7162)",
     "0.5577\n(0.5574–0.5579)", "8,148,638"],

    ["VGG16 (cls)", "0.8761\n(0.8760–0.8762)", "0.9125\n(0.9124–0.9126)", "0.8491\n(0.8490–0.8493)",
     "0.8797\n(0.8796–0.8798)", "0.9460\n(0.9459–0.9460)", "0.8797\n(0.8796–0.8798)",
     "0.7852\n(0.7850–0.7854)", "2,779,150"],

    ["VGG16-UNet", "0.8987\n(0.8987–0.8988)", "0.9504\n(0.9503–0.9505)", "0.8548\n(0.8546–0.8550)",
     "0.9001\n(0.9000–0.9002)", "0.9647\n(0.9646–0.9647)", "0.9001\n(0.9000–0.9002)",
     "0.8183\n(0.8181–0.8184)", "2,674,642"],

    ["ResNet-UNet", "0.8897\n(0.8896–0.8898)", "0.9612\n(0.9611–0.9613)", "0.8266\n(0.8264–0.8267)",
     "0.8888\n(0.8887–0.8889)", "0.9633\n(0.9632–0.9633)", "0.8888\n(0.8887–0.8889)",
     "0.7999\n(0.7997–0.8001)", "8,187,223"],

    ["DenseNet-UNet", "0.8906\n(0.8905–0.8906)", "0.8929\n(0.8928–0.8931)", "0.9030\n(0.9029–0.9032)",
     "0.8980\n(0.8979–0.8980)", "0.9625\n(0.9625–0.9626)", "0.8980\n(0.8979–0.8981)",
     "0.8148\n(0.8147–0.8150)", "1,785,869"]
]

# Create DataFrame
df = pd.DataFrame(data, columns=[
    "Model", "Accuracy", "Precision", "Recall", "F1-score",
    "ROC-AUC", "Dice", "IoU", "False Negatives"
])

# Save to Excel
file_path = "Model_Comparison_Table.xlsx"
df.to_excel(file_path, index=False)

print(f"✅ Excel file saved as '{file_path}'")

In [ ]:
import pandas as pd
from IPython.display import display, HTML

# Define the data with line breaks for confidence intervals
data = [
    [
        "AlexNet",
        "0.7635<br>(0.7633–0.7636)",
        "0.9936<br>(0.9935–0.9936)",
        "0.5602<br>(0.5601–0.5604)",
        "0.7164<br>(0.7163–0.7166)",
        "0.7535<br>(0.7533–0.7536)",
        "0.7165<br>(0.7163–0.7166)",
        "0.5582<br>(0.5580–0.5584)",
        "8,102,319"
    ],
    [
        "DenseNet50",
        "0.8826<br>(0.8825–0.8827)",
        "0.9183<br>(0.9182–0.9184)",
        "0.8561<br>(0.8560–0.8563)",
        "0.8861<br>(0.8860–0.8862)",
        "0.9533<br>(0.9533–0.9534)",
        "0.8861<br>(0.8860–0.8862)",
        "0.7955<br>(0.7954–0.7957)",
        "9,015,355"
    ],
    [
        "UNet (basic)",
        "0.7641<br>(0.7639–0.7642)",
        "1.0000<br>(1.0000–1.0000)",
        "0.5577<br>(0.5575–0.5579)",
        "0.7160<br>(0.7159–0.7162)",
        "0.9256<br>(0.9256–0.9257)",
        "0.7160<br>(0.7159–0.7162)",
        "0.5577<br>(0.5574–0.5579)",
        "8,148,638"
    ],
    [
        "VGG16 (cls)",
        "0.8761<br>(0.8760–0.8762)",
        "0.9125<br>(0.9124–0.9126)",
        "0.8491<br>(0.8490–0.8493)",
        "0.8797<br>(0.8796–0.8798)",
        "0.9460<br>(0.9459–0.9460)",
        "0.8797<br>(0.8796–0.8798)",
        "0.7852<br>(0.7850–0.7854)",
        "2,779,150"
    ],
    [
        "VGG16-UNet",
        "0.8987<br>(0.8987–0.8988)",
        "0.9504<br>(0.9503–0.9505)",
        "0.8548<br>(0.8546–0.8550)",
        "0.9001<br>(0.9000–0.9002)",
        "0.9647<br>(0.9646–0.9647)",
        "0.9001<br>(0.9000–0.9002)",
        "0.8183<br>(0.8181–0.8184)",
        "2,674,642"
    ],
    [
        "ResNet-UNet",
        "0.8897<br>(0.8896–0.8898)",
        "0.9612<br>(0.9611–0.9613)",
        "0.8266<br>(0.8264–0.8267)",
        "0.8888<br>(0.8887–0.8889)",
        "0.9633<br>(0.9632–0.9633)",
        "0.8888<br>(0.8887–0.8889)",
        "0.7999<br>(0.7997–0.8001)",
        "8,187,223"
    ],
    [
        "DenseNet-UNet",
        "0.8906<br>(0.8905–0.8906)",
        "0.8929<br>(0.8928–0.8931)",
        "0.9030<br>(0.9029–0.9032)",
        "0.8980<br>(0.8979–0.8980)",
        "0.9625<br>(0.9625–0.9626)",
        "0.8980<br>(0.8979–0.8981)",
        "0.8148<br>(0.8147–0.8150)",
        "1,785,869"
    ]
]

# Create DataFrame
df = pd.DataFrame(data, columns=[
    "Model", "Accuracy", "Precision", "Recall", "F1-score",
    "ROC-AUC", "Dice", "IoU", "False Negatives"
])

# Display as HTML for better formatting (with line breaks)
html_table = df.to_html(escape=False, index=False)

# Optional: Add some inline CSS for better appearance
styled_html = f"""
<style>
    table {{
        border-collapse: collapse;
        width: 100%;
        font-family: Arial, sans-serif;
        font-size: 12px;
    }}
    th, td {{
        border: 1px solid #999;
        padding: 8px;
        text-align: center;
    }}
    th {{
        background-color: #f0f0f0;
        font-weight: bold;
    }}
    tr:nth-child(even) {{
        background-color: #f9f9f9;
    }}
</style>
{html_table}
"""

# Render in notebook
display(HTML(styled_html))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle
from matplotlib.patches import Circle
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Sample data (replace with your actual classified raster)
# Assume 'classified_map' is a 2D array: 1 = forest (green), 0 = non-forest (red)
np.random.seed(42)
classified_map = np.random.choice([0, 1], size=(100, 100))

fig, ax = plt.subplots(1, 1, figsize=(10, 8), subplot_kw={'projection': ccrs.PlateCarree()})

# Plot the classified map
im = ax.imshow(classified_map, cmap='RdYlGn', origin='upper', extent=[0, 10, 0, 10])

# Add coastlines and borders
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)

# Add scale bar (1 km)
x0, x1 = 8, 9
y0 = 2
scale_bar_length = 1  # in degrees (approx. 111 km at equator; adjust based on lat)
ax.plot([x0, x1], [y0, y0], 'k-', transform=ccrs.PlateCarree())
ax.text(x0, y0 - 0.2, '1 km', transform=ccrs.PlateCarree(), fontsize=10, ha='center')

# Add compass rose (north arrow)
ax.annotate('', xy=(0.05, 0.95), xytext=(0.1, 0.95), arrowprops=dict(arrowstyle='->', color='black'), transform=ax.transAxes)
ax.text(0.07, 0.96, 'N', transform=ax.transAxes, fontsize=10, ha='center')

# Add legend
legend_elements = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='green', markersize=10, label='Forest'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=10, label='Non-Forest')
]
ax.legend(handles=legend_elements, loc='lower right', frameon=True)

# Add NDVI threshold text
ax.text(0.02, 0.98, "NDVI > 0.5 = Forest", transform=ax.transAxes, fontsize=10, verticalalignment='top', bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
ax.text(0.02, 0.95, "NDVI ≤ 0.5 = Non-Forest", transform=ax.transAxes, fontsize=10, verticalalignment='top', bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

# Title and labels
ax.set_title("Forest Cover Classification Using U-Net", fontsize=14, pad=20)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

plt.tight_layout()
plt.show()

In [ ]:
!pip install cartopy

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Mock data: Forest loss map (0=non-forest, 1=forest, 2=loss)
data = np.random.choice([0, 1, 2], size=(100, 100), p=[0.3, 0.6, 0.1])

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(data, cmap=plt.cm.get_cmap('RdYlGn', 3))

# Add legends
cbar = plt.colorbar(im, ticks=[0, 1, 2])
cbar.ax.set_yticklabels(['Non-Forest', 'Forest', 'Loss (ΔNDVI > 0.2)'])
plt.title("Forest Loss Map (2020-2023)")
plt.xlabel("Scale: 1 pixel = 10 m")

# Add scale bar and north arrow (manually)
ax.text(10, 90, "↑ N", fontsize=12, ha='center')
ax.plot([10, 60], [85, 85], 'k-', lw=2)
ax.text(35, 80, "500 m", ha='center')

plt.show()

In [ ]:
# Example: Plotting retained images per year
import matplotlib.pyplot as plt
years = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
retained = [88, 85, 79, 82, 91, 89, 94, 87, 93]  # Example counts
plt.bar(years, retained, color='skyblue')
plt.xlabel("Year"); plt.ylabel("Retained Images"); plt.title("Temporal Coverage Post-Exclusion")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


metrics = ['Accuracy', 'F1-score', 'Dice', 'IoU']

full_dataset = [0.8157, 0.8171, 0.8171, 0.7153]

# Cloud-free dataset (actual performance of VGG16-UNet)
cloud_free_dataset = [0.8987, 0.9001, 0.9001, 0.8183]

# Convert to percentages for clearer visualization
full_dataset_pct = [x * 100 for x in full_dataset]
cloud_free_dataset_pct = [x * 100 for x in cloud_free_dataset]

x = np.arange(len(metrics))  # label locations
width = 0.35  # width of bars

# Create the plot
fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(x - width/2, full_dataset_pct, width, label='Full Dataset (with Cloudy Images)', color='skyblue', edgecolor='black', alpha=0.9)
bars2 = ax.bar(x + width/2, cloud_free_dataset_pct, width, label='Cloud-Free Dataset', color='mediumseagreen', edgecolor='black', alpha=0.9)

# Add labels, title, and legend
ax.set_xlabel('Performance Metrics', fontsize=12)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Full vs. Cloud-Free Dataset', fontsize=14, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 100)

# Add value labels on top of bars
def add_labels(bars):
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.1f}%',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=10)

add_labels(bars1)
add_labels(bars2)

ax.legend(loc='lower right', fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.5)

# Improve layout and display
plt.tight_layout()

# Optional: Save the figure
# plt.savefig("figureY_sensitivity_analysis.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Metrics
metrics = ['Accuracy', 'F1-score', 'Dice', 'IoU']

# Full dataset (estimated with cloudy images)
full_dataset = [0.8157, 0.8171, 0.8171, 0.7153]

# Cloud-free dataset (actual VGG16-UNet)
cloud_free_dataset = [0.8987, 0.9001, 0.9001, 0.8183]

# Convert to percentages
full_pct = [x * 100 for x in full_dataset]
cf_pct = [x * 100 for x in cloud_free_dataset]

# Plot
x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(x - width/2, full_pct, width,
               label='Full Dataset\n(with cloudy images)',
               color='lightcoral', edgecolor='black', alpha=0.85)

bars2 = ax.bar(x + width/2, cf_pct, width,
               label='Cloud-Free Dataset',
               color='steelblue', edgecolor='black', alpha=0.85)

ax.set_xlabel('Performance Metrics', fontsize=12)
ax.set_ylabel('Score (%)', fontsize=12)
ax.set_title('Sensitivity Analysis – Full vs. Cloud-Free Dataset', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 100)

def add_labels(bars):
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.1f}%',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=10)

add_labels(bars1)
add_labels(bars2)

ax.legend(loc='lower right', fontsize=11, frameon=True)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score
)
import tensorflow as tf

# ==============================
# Parameters
# ==============================
K = 5
random_state = 42
batch_size = 32
epochs = 100
size = 256  # image size

# Normalize input images
tiles_img_norm = tiles_img / 255.0
# Expand mask dims if needed: (N, H, W) -> (N, H, W, 1)
tiles_mask = tiles_mask[..., np.newaxis]

# Initialize K-Fold
kf = KFold(n_splits=K, shuffle=True, random_state=random_state)

# Store metrics per fold
metrics_per_fold = {
    "Accuracy": [], "Precision": [], "Recall": [], "F1": [],
    "Dice": [], "mIoU": [], "ROC-AUC": []
}
conf_matrices = []

# ==============================
# Learning Rate Scheduler
# ==============================
def lr_schedule(epoch):
    lr = 1e-4
    if epoch > 70:
        lr *= 0.1
    elif epoch > 50:
        lr *= 0.5
    return lr

# ==============================
# Start Cross-Validation
# ==============================
fold = 1
for train_index, val_index in kf.split(tiles_img_norm):
    print(f"\n===== Fold {fold}/{K} =====")
    
    # Split data
    X_train_cv, X_val_cv = tiles_img_norm[train_index], tiles_img_norm[val_index]
    y_train_cv, y_val_cv = tiles_mask[train_index], tiles_mask[val_index]
    
    # Data augmentation
    datagen_cv = tf.keras.preprocessing.image.ImageDataGenerator(
        shear_range=0.2, zoom_range=0.2, horizontal_flip=True,
        rotation_range=20, width_shift_range=0.2, height_shift_range=0.2,
        brightness_range=[0.8, 1.2]
    )
    
    # Create model (replace with your actual function)
    model_cv = vgg16_unet_model(input_size=(size, size, 3), freeze_encoder=True)
    
    # Compile model
    model_cv.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    # Callbacks
    lr_scheduler_cv = tf.keras.callbacks.LearningRateScheduler(lr_schedule)
    early_stop_cv = tf.keras.callbacks.EarlyStopping(
        monitor='val_dice_coef', patience=5, mode='max', restore_best_weights=True, verbose=1
    )
    
    # Train model
    print(f"Training Fold {fold}...")
    model_cv.fit(
        datagen_cv.flow(X_train_cv, y_train_cv, batch_size=batch_size),
        epochs=epochs,
        validation_data=(X_val_cv, y_val_cv),
        callbacks=[lr_scheduler_cv, early_stop_cv],
        verbose=1
    )
    
    # Predict
    y_prob = model_cv.predict(X_val_cv, verbose=0)
    y_pred = (y_prob > 0.5).astype(np.uint8)
    
    # Flatten for metric computation
    y_true_flat = y_val_cv.flatten()
    y_pred_flat = y_pred.flatten()
    y_prob_flat = y_prob.flatten()
    
    # Compute metrics
    acc = accuracy_score(y_true_flat, y_pred_flat)
    prec = precision_score(y_true_flat, y_pred_flat, zero_division=0)
    rec = recall_score(y_true_flat, y_pred_flat, zero_division=0)
    f1 = f1_score(y_true_flat, y_pred_flat, zero_division=0)
    
    # Dice Coefficient (same as F1 for binary, but standard in segmentation)
    intersection = np.logical_and(y_true_flat, y_pred_flat).sum()
    dice = 2. * intersection / (y_true_flat.sum() + y_pred_flat.sum() + 1e-8)
    
    # mIoU
    union = np.logical_or(y_true_flat, y_pred_flat).sum()
    miou = intersection / union if union > 0 else 0.0
    
    # ROC-AUC
    roc = roc_auc_score(y_true_flat, y_prob_flat)
    
    # Store metrics
    metrics_per_fold["Accuracy"].append(acc)
    metrics_per_fold["Precision"].append(prec)
    metrics_per_fold["Recall"].append(rec)
    metrics_per_fold["F1"].append(f1)
    metrics_per_fold["Dice"].append(dice)
    metrics_per_fold["mIoU"].append(miou)
    metrics_per_fold["ROC-AUC"].append(roc)
    
    # Confusion matrix
    cm = confusion_matrix(y_true_flat, y_pred_flat)
    conf_matrices.append(cm)
    
    # Print fold results
    print(f"Fold {fold} Results:")
    print(f"  Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}")
    print(f"  F1: {f1:.4f}, Dice: {dice:.4f}, mIoU: {miou:.4f}, ROC-AUC: {roc:.4f}")
    
    fold += 1

# ==============================
# Final Results Summary
# ==============================
print("\n" + "="*70)
print("            CROSS-VALIDATION RESULTS (5-FOLD)")
print("="*70)

results_summary = {}
for metric, values in metrics_per_fold.items():
    mean_val = np.mean(values)
    std_val = np.std(values)
    ci95 = 1.96 * (std_val / np.sqrt(K))  # Approximate 95% CI
    ci_lower = mean_val - ci95
    ci_upper = mean_val + ci95
    results_summary[metric] = (mean_val, std_val, ci_lower, ci_upper)
    print(f"{metric:<10}: {mean_val:.4f} ± {std_val:.4f} | 95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")

print("="*70)

# ==============================
# Per-Fold Results Table
# ==============================
results_df = pd.DataFrame({
    'Fold': [f"F{i}" for i in range(1, K+1)],
    'Accuracy': [f"{v:.4f}" for v in metrics_per_fold["Accuracy"]],
    'Precision': [f"{v:.4f}" for v in metrics_per_fold["Precision"]],
    'Recall': [f"{v:.4f}" for v in metrics_per_fold["Recall"]],
    'F1': [f"{v:.4f}" for v in metrics_per_fold["F1"]],
    'Dice': [f"{v:.4f}" for v in metrics_per_fold["Dice"]],
    'mIoU': [f"{v:.4f}" for v in metrics_per_fold["mIoU"]],
    'ROC-AUC': [f"{v:.4f}" for v in metrics_per_fold["ROC-AUC"]]
})
print("\nPer-Fold Results:")
print(results_df.to_string(index=False))

# Optional: Save to CSV
# results_df.to_csv("cv_per_fold_results.csv", index=False)

# ==============================
# Plot: Metric Stability Across Folds
# ==============================
plt.figure(figsize=(12, 6))
metrics_to_plot = ["Accuracy", "Dice", "mIoU", "F1", "Recall"]
x_folds = np.arange(1, K+1)

for metric in metrics_to_plot:
    plt.plot(x_folds, metrics_per_fold[metric], 'o-', label=metric)

plt.axhline(np.mean(metrics_per_fold["Dice"]), color='gray', linestyle='--', alpha=0.7,
            label=f"Mean Dice = {np.mean(metrics_per_fold['Dice']):.4f}")
plt.xlabel("Fold", fontsize=12)
plt.ylabel("Score", fontsize=12)
plt.title("Cross-Validation Performance per Fold", fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0.8, 1.0)
plt.xticks(x_folds)
plt.tight_layout()
plt.show()

# ==============================
# Average Confusion Matrix
# ==============================
avg_cm = np.mean(conf_matrices, axis=0)
plt.figure(figsize=(6, 5))
sns.heatmap(avg_cm, annot=True, fmt=".0f", cmap="Blues", cbar=True,
            xticklabels=["Non-Forest", "Forest"],
            yticklabels=["Non-Forest", "Forest"], square=True)
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.title("Average Confusion Matrix (5-Fold CV)", fontsize=13)
plt.tight_layout()
plt.show()

# Optional: Save confusion matrix
# plt.savefig("avg_confusion_matrix_cv.png", dpi=300, bbox_inches='tight')

In [ ]:
#K FOLD CROSS VALIDATION

In [ ]:
import numpy as np
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score
)
import tensorflow as tf
from tensorflow.keras import backend as K

# ==============================
# Parameters
# ==============================
K = 5
random_state = 42
batch_size = 32
epochs = 20
size = 256  # image size

# Normalize input images
tiles_img_norm = tiles_img / 255.0
# Expand mask dims: (N, H, W) -> (N, H, W, 1)
tiles_mask = tiles_mask[..., np.newaxis]

# Initialize K-Fold
kf = KFold(n_splits=K, shuffle=True, random_state=random_state)

# Store metrics per fold
metrics_per_fold = {
    "Accuracy": [], "Precision": [], "Recall": [], "F1": [],
    "Dice": [], "mIoU": [], "ROC-AUC": []
}
conf_matrices = []

# ==============================
# Dice Coefficient Metric
# ==============================




def dice_coef(y_true, y_pred):
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + 1e-8) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + 1e-8)
# ==============================
# Learning Rate Scheduler (on Dice)
# ==============================
lr_scheduler_cv = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_dice_coef',
    factor=0.5,
    patience=3,
    mode='max',
    min_lr=1e-7,
    verbose=1
)

# ==============================
# Start Cross-Validation
# ==============================
fold = 1
for train_index, val_index in kf.split(tiles_img_norm):
    print(f"\n===== Fold {fold}/{K} =====")
    
    # Split data
    X_train_cv, X_val_cv = tiles_img_norm[train_index], tiles_img_norm[val_index]
    y_train_cv, y_val_cv = tiles_mask[train_index], tiles_mask[val_index]
    
    # Data augmentation
    datagen_cv = tf.keras.preprocessing.image.ImageDataGenerator(
        shear_range=0.2, zoom_range=0.2, horizontal_flip=True,
        rotation_range=20, width_shift_range=0.2, height_shift_range=0.2,
        brightness_range=[0.8, 1.2]
    )
    
    # Create model (replace with your actual function)
    model_cv = vgg16_unet_model(input_size=(size, size, 3), freeze_encoder=True)
    
    # Compile model with Dice as metric
    model_cv.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy', dice_coef]
    )
    
    # Callbacks: Early stopping and checkpoint on Dice
    early_stop_cv = tf.keras.callbacks.EarlyStopping(
        monitor='val_dice_coef',
        patience=5,
        mode='max',
        restore_best_weights=True,
        verbose=1
    )
    
    model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
        f'best_vgg16_unet_fold_{fold}.h5',
        monitor='val_dice_coef',
        save_best_only=True,
        mode='max',
        verbose=1
    )
    
    # Train model
    print(f"Training Fold {fold}...")
    model_cv.fit(
        datagen_cv.flow(X_train_cv, y_train_cv, batch_size=batch_size),
        epochs=epochs,
        validation_data=(X_val_cv, y_val_cv),
        callbacks=[lr_scheduler_cv, early_stop_cv, model_checkpoint],
        verbose=1
    )
    
    # Predict
    y_prob = model_cv.predict(X_val_cv, verbose=0)
    y_pred = (y_prob > 0.5).astype(np.uint8)
    
    # Flatten for metric computation
    y_true_flat = y_val_cv.flatten()
    y_pred_flat = y_pred.flatten()
    y_prob_flat = y_prob.flatten()
    
    # Compute metrics
    acc = accuracy_score(y_true_flat, y_pred_flat)
    prec = precision_score(y_true_flat, y_pred_flat, zero_division=0)
    rec = recall_score(y_true_flat, y_pred_flat, zero_division=0)
    f1 = f1_score(y_true_flat, y_pred_flat, zero_division=0)
    
    # Dice Coefficient
    intersection = np.logical_and(y_true_flat, y_pred_flat).sum()
    dice = 2. * intersection / (y_true_flat.sum() + y_pred_flat.sum() + 1e-8)
    
    # mIoU
    union = np.logical_or(y_true_flat, y_pred_flat).sum()
    miou = intersection / union if union > 0 else 0.0
    
    # ROC-AUC
    roc = roc_auc_score(y_true_flat, y_prob_flat)
    
    # Store metrics
    metrics_per_fold["Accuracy"].append(acc)
    metrics_per_fold["Precision"].append(prec)
    metrics_per_fold["Recall"].append(rec)
    metrics_per_fold["F1"].append(f1)
    metrics_per_fold["Dice"].append(dice)
    metrics_per_fold["mIoU"].append(miou)
    metrics_per_fold["ROC-AUC"].append(roc)
    
    # Confusion matrix
    cm = confusion_matrix(y_true_flat, y_pred_flat)
    conf_matrices.append(cm)
    
    # Print fold results
    print(f"Fold {fold} Results:")
    print(f"  Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}")
    print(f"  F1: {f1:.4f}, Dice: {dice:.4f}, mIoU: {miou:.4f}, ROC-AUC: {roc:.4f}")
    
    fold += 1

# ==============================
# Final Results Summary
# ==============================
print("\n" + "="*70)
print("            CROSS-VALIDATION RESULTS (5-FOLD)")
print("="*70)

results_summary = {}
for metric, values in metrics_per_fold.items():
    mean_val = np.mean(values)
    std_val = np.std(values)
    ci95 = 1.96 * (std_val / np.sqrt(K))  # Approximate 95% CI
    ci_lower = mean_val - ci95
    ci_upper = mean_val + ci95
    results_summary[metric] = (mean_val, std_val, ci_lower, ci_upper)
    print(f"{metric:<10}: {mean_val:.4f} ± {std_val:.4f} | 95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")

print("="*70)

# ==============================
# Per-Fold Results Table
# ==============================
results_df = pd.DataFrame({
    'Fold': [f"F{i}" for i in range(1, K+1)],
    'Accuracy': [f"{v:.4f}" for v in metrics_per_fold["Accuracy"]],
    'Precision': [f"{v:.4f}" for v in metrics_per_fold["Precision"]],
    'Recall': [f"{v:.4f}" for v in metrics_per_fold["Recall"]],
    'F1': [f"{v:.4f}" for v in metrics_per_fold["F1"]],
    'Dice': [f"{v:.4f}" for v in metrics_per_fold["Dice"]],
    'mIoU': [f"{v:.4f}" for v in metrics_per_fold["mIoU"]],
    'ROC-AUC': [f"{v:.4f}" for v in metrics_per_fold["ROC-AUC"]]
})
print("\nPer-Fold Results:")
print(results_df.to_string(index=False))

# Optional: Save to CSV
# results_df.to_csv("cv_per_fold_results.csv", index=False)

# ==============================
# Plot: Metric Stability Across Folds
# ==============================
plt.figure(figsize=(12, 6))
metrics_to_plot = ["Accuracy", "Dice", "mIoU", "F1", "Recall"]
x_folds = np.arange(1, K+1)

for metric in metrics_to_plot:
    plt.plot(x_folds, metrics_per_fold[metric], 'o-', label=metric)

plt.axhline(np.mean(metrics_per_fold["Dice"]), color='gray', linestyle='--', alpha=0.7,
            label=f"Mean Dice = {np.mean(metrics_per_fold['Dice']):.4f}")
plt.xlabel("Fold", fontsize=12)
plt.ylabel("Score", fontsize=12)
plt.title("Cross-Validation Performance per Fold", fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0.8, 1.0)
plt.xticks(x_folds)
plt.tight_layout()
plt.show()

# ==============================
# Average Confusion Matrix
# ==============================
avg_cm = np.mean(conf_matrices, axis=0)
plt.figure(figsize=(6, 5))
sns.heatmap(avg_cm, annot=True, fmt=".0f", cmap="Blues", cbar=True,
            xticklabels=["Non-Forest", "Forest"],
            yticklabels=["Non-Forest", "Forest"], square=True)
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.title("Average Confusion Matrix (5-Fold CV)", fontsize=13)
plt.tight_layout()
plt.show()

# Optional: Save results
# plt.savefig("avg_confusion_matrix_cv.png", dpi=300, bbox_inches='tight')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Metrics
metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'Dice', 'mIoU', 'ROC-AUC']

# DenseNet-U-Net mean values
densenet_mean = [0.8935, 0.9332, 0.8713, 0.9003, 0.9003, 0.8187, 0.9637]
# DenseNet-U-Net std
densenet_std = [0.0022, 0.0289, 0.0296, 0.0046, 0.0046, 0.0076, 0.0034]

# VGG16-U-Net mean values
vgg16_mean = [0.8939, 0.9348, 0.8705, 0.9004, 0.9004, 0.8189, 0.9644]
# VGG16-U-Net std
vgg16_std = [0.0075, 0.0308, 0.0353, 0.0102, 0.0102, 0.0169, 0.0045]

x = np.arange(len(metrics))
width = 0.35  # bar width

plt.figure(figsize=(12,6))
plt.bar(x - width/2, densenet_mean, width, yerr=densenet_std, capsize=5, label='DenseNet-U-Net', color='skyblue')
plt.bar(x + width/2, vgg16_mean, width, yerr=vgg16_std, capsize=5, label='VGG16-U-Net', color='orange')

plt.xticks(x, metrics, rotation=45)
plt.ylabel('Score')
plt.ylim(0.75, 1.0)
plt.title('5-Fold Cross-Validation: DenseNet-U-Net vs VGG16-U-Net')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score
)
from tensorflow.keras import backend as K

# ==============================
# Parameters
# ==============================
K = 10  # Changed from 5 to 10
random_state = 42
batch_size = 32
epochs = 20
size = 256  # image size

# Normalize input images
tiles_img_norm = tiles_img / 255.0
# Expand mask dims: (N, H, W) -> (N, H, W, 1)
tiles_mask = tiles_mask[..., np.newaxis]

# Initialize 10-Fold Cross-Validator
kf = KFold(n_splits=K, shuffle=True, random_state=random_state)

# Store metrics per fold
metrics_per_fold = {
    "Accuracy": [], "Precision": [], "Recall": [], "F1": [],
    "Dice": [], "mIoU": [], "ROC-AUC": []
}
conf_matrices = []

# ==============================
# Dice Coefficient Metric
# ==============================
def dice_coef(y_true, y_pred):
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + 1e-8) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + 1e-8)

# ==============================
# Learning Rate Scheduler
# ==============================
lr_scheduler_cv = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_dice_coef',
    factor=0.5,
    patience=2,
    mode='max',
    min_lr=1e-7,
    verbose=1
)

# ==============================
# Start 10-Fold Cross-Validation
# ==============================
fold = 1
for train_index, val_index in kf.split(tiles_img_norm):
    print(f"\n===== Fold {fold}/{K} =====")
    
    # Split data
    X_train_cv, X_val_cv = tiles_img_norm[train_index], tiles_img_norm[val_index]
    y_train_cv, y_val_cv = tiles_mask[train_index], tiles_mask[val_index]
    
    # Data augmentation
    datagen_cv = tf.keras.preprocessing.image.ImageDataGenerator(
        shear_range=0.2, zoom_range=0.2, horizontal_flip=True,
        rotation_range=20, width_shift_range=0.2, height_shift_range=0.2,
        brightness_range=[0.8, 1.2]
    )
    
    # Create model (replace with your actual function)
    model_cv = vgg16_unet_model(input_size=(size, size, 3), freeze_encoder=True)
    
    # Compile model
    model_cv.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy', dice_coef]
    )
    
    # Callbacks
    early_stop_cv = tf.keras.callbacks.EarlyStopping(
        monitor='val_dice_coef',
        patience=5,
        mode='max',
        restore_best_weights=True,
        verbose=1
    )
    
    model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
        f'best_10kf_vgg16_unet_fold_{fold}.h5',
        monitor='val_dice_coef',
        save_best_only=True,
        mode='max',
        verbose=1
    )
    
    # Train model
    print(f"Training Fold {fold}...")
    model_cv.fit(
        datagen_cv.flow(X_train_cv, y_train_cv, batch_size=batch_size),
        epochs=epochs,
        validation_data=(X_val_cv, y_val_cv),
        callbacks=[lr_scheduler_cv, early_stop_cv, model_checkpoint],
        verbose=1
    )
    
    # Predict
    y_prob = model_cv.predict(X_val_cv, verbose=0)
    y_pred = (y_prob > 0.5).astype(np.uint8)
    
    # Flatten for metric computation
    y_true_flat = y_val_cv.flatten()
    y_pred_flat = y_pred.flatten()
    y_prob_flat = y_prob.flatten()
    
    # Compute metrics
    acc = accuracy_score(y_true_flat, y_pred_flat)
    prec = precision_score(y_true_flat, y_pred_flat, zero_division=0)
    rec = recall_score(y_true_flat, y_pred_flat, zero_division=0)
    f1 = f1_score(y_true_flat, y_pred_flat, zero_division=0)
    
    # Dice Coefficient (NumPy version for consistency)
    intersection = np.logical_and(y_true_flat, y_pred_flat).sum()
    dice = 2. * intersection / (y_true_flat.sum() + y_pred_flat.sum() + 1e-8)
    
    # mIoU
    union = np.logical_or(y_true_flat, y_pred_flat).sum()
    miou = intersection / union if union > 0 else 0.0
    
    # ROC-AUC
    roc = roc_auc_score(y_true_flat, y_prob_flat)
    
    # Store metrics
    metrics_per_fold["Accuracy"].append(acc)
    metrics_per_fold["Precision"].append(prec)
    metrics_per_fold["Recall"].append(rec)
    metrics_per_fold["F1"].append(f1)
    metrics_per_fold["Dice"].append(dice)
    metrics_per_fold["mIoU"].append(miou)
    metrics_per_fold["ROC-AUC"].append(roc)
    
    # Confusion matrix
    cm = confusion_matrix(y_true_flat, y_pred_flat)
    conf_matrices.append(cm)
    
    # Print fold results
    print(f"Fold {fold} Results:")
    print(f"  Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}")
    print(f"  F1: {f1:.4f}, Dice: {dice:.4f}, mIoU: {miou:.4f}, ROC-AUC: {roc:.4f}")
    
    fold += 1

# ==============================
# Final Results Summary (10-Fold)
# ==============================
print("\n" + "="*70)
print("            CROSS-VALIDATION RESULTS (10-FOLD)")
print("="*70)

results_summary = {}
for metric, values in metrics_per_fold.items():
    mean_val = np.mean(values)
    std_val = np.std(values)
    ci95 = 1.96 * (std_val / np.sqrt(K))  # 95% CI using standard error
    ci_lower = mean_val - ci95
    ci_upper = mean_val + ci95
    results_summary[metric] = (mean_val, std_val, ci_lower, ci_upper)
    print(f"{metric:<10}: {mean_val:.4f} ± {std_val:.4f} | 95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")

print("="*70)

# ==============================
# Per-Fold Results Table
# ==============================
results_df = pd.DataFrame({
    'Fold': [f"F{i}" for i in range(1, K+1)],
    'Accuracy': [f"{v:.4f}" for v in metrics_per_fold["Accuracy"]],
    'Precision': [f"{v:.4f}" for v in metrics_per_fold["Precision"]],
    'Recall': [f"{v:.4f}" for v in metrics_per_fold["Recall"]],
    'F1': [f"{v:.4f}" for v in metrics_per_fold["F1"]],
    'Dice': [f"{v:.4f}" for v in metrics_per_fold["Dice"]],
    'mIoU': [f"{v:.4f}" for v in metrics_per_fold["mIoU"]],
    'ROC-AUC': [f"{v:.4f}" for v in metrics_per_fold["ROC-AUC"]]
})
print("\nPer-Fold Results:")
print(results_df.to_string(index=False))

# Optional: Save to CSV
# results_df.to_csv("cv_per_fold_results_10fold.csv", index=False)

# ==============================
# Plot: Metric Stability Across Folds
# ==============================
plt.figure(figsize=(14, 7))
metrics_to_plot = ["Accuracy", "F1", "Dice", "mIoU", "Recall", "ROC-AUC"]
x_folds = np.arange(1, K+1)

for metric in metrics_to_plot:
    plt.plot(x_folds, metrics_per_fold[metric], 'o-', label=metric)

# Add mean lines
for metric in metrics_to_plot:
    mean_val = np.mean(metrics_per_fold[metric])
    plt.axhline(mean_val, color=plt.gca().get_lines()[-1].get_color(),
                linestyle='--', alpha=0.6)

plt.xlabel("Fold", fontsize=12)
plt.ylabel("Score", fontsize=12)
plt.title("10-Fold Cross-Validation Performance", fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0.75, 1.0)
plt.xticks(x_folds)
plt.tight_layout()
plt.show()

# ==============================
# Average Confusion Matrix
# ==============================
avg_cm = np.mean(conf_matrices, axis=0)
plt.figure(figsize=(6, 5))
sns.heatmap(avg_cm, annot=True, fmt=".0f", cmap="Blues", cbar=True,
            xticklabels=["Non-Forest", "Forest"],
            yticklabels=["Non-Forest", "Forest"], square=True)
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.title("Average Confusion Matrix (10-Fold CV)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from scipy.stats import ttest_rel, wilcoxon
import pandas as pd

# Example per-fold results
results = {
    'Accuracy': {'DenseNet': [0.8929, 0.8899, 0.8933, 0.8968, 0.8945],
                 'VGG16':   [0.8846, 0.8869, 0.9019, 0.8932, 0.9030]},
    'Precision': {'DenseNet': [0.9092, 0.9479, 0.8908, 0.9480, 0.9701],
                  'VGG16':   [0.8911, 0.9667, 0.9053, 0.9626, 0.9485]},
    'Recall': {'DenseNet': [0.8879, 0.8431, 0.9199, 0.8644, 0.8414],
               'VGG16':   [0.8928, 0.8194, 0.9188, 0.8434, 0.8779]},
    'F1': {'DenseNet': [0.8984, 0.8924, 0.9051, 0.9043, 0.9011],
           'VGG16':   [0.8919, 0.8870, 0.9120, 0.8990, 0.9119]},
    'Dice': {'DenseNet': [0.8984, 0.8924, 0.9051, 0.9043, 0.9011],
             'VGG16':   [0.8919, 0.8870, 0.9120, 0.8990, 0.9119]},
    'mIoU': {'DenseNet': [0.8156, 0.8058, 0.8266, 0.8253, 0.8201],
             'VGG16':   [0.8050, 0.7969, 0.8382, 0.8166, 0.8380]},
    'ROC-AUC': {'DenseNet': [0.9595, 0.9609, 0.9677, 0.9625, 0.9677],
                'VGG16':   [0.9580, 0.9615, 0.9712, 0.9646, 0.9669]}
}

# Function to annotate significance
def significance(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return ''

# Initialize dataframe
metrics = list(results.keys())
df_stats = pd.DataFrame(columns=['Metric', 't-stat', 'p-value (t-test)', 't-significance', 
                                 'W-stat', 'p-value (Wilcoxon)', 'W-significance'])

# Perform statistical tests
for metric in metrics:
    d_values = np.array(results[metric]['DenseNet'])
    v_values = np.array(results[metric]['VGG16'])
    
    # Paired t-test
    t_stat, t_pval = ttest_rel(d_values, v_values)
    
    # Wilcoxon signed-rank test
    w_stat, w_pval = wilcoxon(d_values, v_values)
    
    df_stats = df_stats.append({
        'Metric': metric,
        't-stat': round(t_stat, 3),
        'p-value (t-test)': round(t_pval, 4),
        't-significance': significance(t_pval),
        'W-stat': round(w_stat, 3),
        'p-value (Wilcoxon)': round(w_pval, 4),
        'W-significance': significance(w_pval)
    }, ignore_index=True)

# Display results
print(df_stats)
